# 05. Noise Mitigation Experiments

This notebook evaluates selected methods for mitigating the impact of synthetic label noise on multi-label chest X-ray classification. The experiments apply each mitigation method across the configured symmetric and asymmetric noise conditions.

**Methods evaluated:** label smoothing, MixUp, co-teaching, and confident learning.


In [ ]:
import os
import gc
import time
import json
import ctypes
import cleanlab
from datetime import datetime
from copy import deepcopy
from cleanlab.filter import find_label_issues

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from torchvision import transforms, models
from sklearn.metrics import roc_auc_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

VRAM_THRESHOLD_GB = 12.0

print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {vram_gb:.2f} GB")

Device: cuda
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
VRAM: 7.96 GiB


## 1. Configuration and Experiment Setup

Define data paths, training parameters, target pathologies, mitigation methods, noise conditions, random seeds, and output directories.


In [ ]:
RAW_DIR = os.path.join('..', 'data', 'raw')
PROCESSED_DIR = os.path.join('..', 'data', 'processed')
TEST_DIR = os.path.join('..', 'data', 'test')
MODELS_DIR = os.path.join('..', 'models', 'mitigation_experiments')
RESULTS_DIR = os.path.join('..', 'results', 'mitigation_experiments')
LOG_DIR = os.path.join('..', 'results', 'logs')
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

TARGET_PATHOLOGIES = [
    'Pleural Effusion',
    'Consolidation',
    'Cardiomegaly', 
    'Atelectasis', 
    'Edema'
]

TRAIN_CONFIG = {
    'batch_size': 32,
    'num_workers': 0,
    'learning_rate': 1e-4,
    'num_epochs': 5,
    'image_size': 224,
    'weight_decay': 1e-5,
    'early_stopping_patience': 2,
}

MITIGATION_CONFIG = {
    'label_smoothing_epsilon': 0.1,       # Smoothing factor for label smoothing
    'mixup_alpha': 0.2,                    # Beta distribution parameter for MixUp
    'coteaching_forget_rate': 0.2,         # Fraction of large-loss samples to discard
    'coteaching_warmup_epochs': 1,         # Epochs before starting sample selection
    'confident_learning_prune_ratio': 0.1, # Fraction of samples cleanlab may remove
}

NOISE_CONFIGS = [
    # 5% and 10% noise levels are commented out to reduce the number of experiments
    # ('symmetric', 0.05), ('symmetric', 0.10), 
    ('symmetric', 0.20), ('symmetric', 0.40),
    # ('asymmetric', 0.05), ('asymmetric', 0.10),
    ('asymmetric', 0.20), ('asymmetric', 0.40),
]

if vram_gb >= VRAM_THRESHOLD_GB:
    MITIGATION_METHODS = ['co_teaching', 'confident_learning', 'mixup', 'label_smoothing']
else:
    MITIGATION_METHODS = ['label_smoothing', 'mixup', 'confident_learning']

SEEDS = [132, 456, 789]
COOLDOWN_SECONDS = 180

print(f"Mitigation methods: {len(MITIGATION_METHODS)}")
print(f"Noise configurations: {len(NOISE_CONFIGS)}")
print(f"Seeds per config: {len(SEEDS)}")
print(f"Total experiments: {len(MITIGATION_METHODS) * len(NOISE_CONFIGS) * len(SEEDS)}")
print(json.dumps(MITIGATION_CONFIG, indent=2))

Mitigation methods: 3
Noise configurations: 8
Seeds per config: 3
Total experiments: 72
{
  "label_smoothing_epsilon": 0.1,
  "mixup_alpha": 0.2,
  "coteaching_forget_rate": 0.2,
  "coteaching_warmup_epochs": 1,
  "confident_learning_prune_ratio": 0.1
}


## 2. Prevent System Sleep During Long Runs

Define utilities to keep the system awake during long mitigation experiment sessions and restore normal sleep behaviour afterwards.


In [17]:
ES_CONTINUOUS = 0x80000000
ES_SYSTEM_REQUIRED = 0x00000001
ES_DISPLAY_REQUIRED = 0x00000002

def prevent_sleep():
    try:
        ctypes.windll.kernel32.SetThreadExecutionState(
            ES_CONTINUOUS | ES_SYSTEM_REQUIRED | ES_DISPLAY_REQUIRED
        )
        print("Sleep prevention ENABLED")
    except Exception as e:
        print(f"Could not prevent sleep: {e}")

def allow_sleep():
    try:
        ctypes.windll.kernel32.SetThreadExecutionState(ES_CONTINUOUS)
        print("Sleep prevention DISABLED")
    except Exception as e:
        print(f"Could not restore sleep: {e}")

prevent_sleep()

Sleep prevention ENABLED


## 3. Dataset and Data Loading

Define the dataset class, image transformations, and optional sample-index handling required by the mitigation methods.


In [18]:
class CheXpertDataset(Dataset):
    def __init__(self, csv_path, data_dir, pathologies, transform=None, return_index=False):
        self.df = pd.read_csv(csv_path)
        self.data_dir = data_dir
        self.pathologies = pathologies
        self.transform = transform
        self.return_index = return_index
        self.labels = self.df[pathologies].values.astype(np.float32)
        self.paths = self.df['Path'].values
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        img_path = os.path.join(self.data_dir, self.paths[idx])
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        labels = torch.from_numpy(self.labels[idx])
        if self.return_index:
            return image, labels, idx
        return image, labels

train_transform = transforms.Compose([
    transforms.Resize((TRAIN_CONFIG['image_size'], TRAIN_CONFIG['image_size'])),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
eval_transform = transforms.Compose([
    transforms.Resize((TRAIN_CONFIG['image_size'], TRAIN_CONFIG['image_size'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_dataset = CheXpertDataset(
    csv_path=os.path.join(PROCESSED_DIR, 'val_clean.csv'),
    data_dir=RAW_DIR, pathologies=TARGET_PATHOLOGIES, transform=eval_transform
)
test_dataset = CheXpertDataset(
    csv_path=os.path.join(PROCESSED_DIR, 'test_official.csv'),
    data_dir=TEST_DIR, pathologies=TARGET_PATHOLOGIES, transform=eval_transform
)
val_loader = DataLoader(val_dataset, batch_size=TRAIN_CONFIG['batch_size'],
                        shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=TRAIN_CONFIG['batch_size'],
                         shuffle=False, num_workers=0, pin_memory=True)
print(f"Val: {len(val_dataset)}, Test: {len(test_dataset)}")

Val: 202, Test: 518


## 4. DenseNet-121 Model Definition

Initialise the common ImageNet-pretrained DenseNet-121 architecture used across mitigation experiments.


In [ ]:
def build_densenet121(num_classes, pretrained=True):
    weights = models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None
    model = models.densenet121(weights=weights)
    model.classifier = nn.Linear(model.classifier.in_features, num_classes)
    return model


@torch.no_grad()
def evaluate(model, loader, criterion, device, pathologies):
    model.eval()
    total_loss = 0.0
    all_logits = []
    all_labels = []
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with autocast('cuda', enabled=device.type == 'cuda'):
            logits = model(images)
            loss = criterion(logits, labels)
        total_loss += loss.item()
        all_logits.append(logits.float().cpu().numpy())
        all_labels.append(labels.cpu().numpy())
    all_logits = np.concatenate(all_logits)
    all_labels = np.concatenate(all_labels)
    all_probs = 1 / (1 + np.exp(-all_logits))
    aucs = {}
    for i, path in enumerate(pathologies):
        if len(np.unique(all_labels[:, i])) > 1:
            aucs[path] = float(roc_auc_score(all_labels[:, i], all_probs[:, i]))
        else:
            aucs[path] = float('nan')
    mean_auc = float(np.nanmean(list(aucs.values())))
    return total_loss / len(loader), mean_auc, aucs


def compute_secondary_metrics_from_model(model, test_loader, device, pathologies, threshold=0.5):
    """Run inference and compute all secondary metrics."""
    model.eval()
    all_logits = []
    all_labels = []
    
    for images, labels in test_loader:
        images = images.to(device, non_blocking=True)
        with autocast('cuda', enabled=device.type == 'cuda'):
            logits = model(images)
        all_logits.append(logits.float().cpu().numpy())
        all_labels.append(labels.numpy())
    
    all_logits = np.concatenate(all_logits)
    all_labels = np.concatenate(all_labels)
    all_probs = 1 / (1 + np.exp(-all_logits))
    all_preds = (all_probs >= threshold).astype(int)
    
    per_pathology = {}
    f1_list, f2_list, sens_list, spec_list, prec_list = [], [], [], [], []
    
    for i, path in enumerate(pathologies):
        yt = all_labels[:, i].astype(int)
        yp = all_preds[:, i]
        
        if len(np.unique(yt)) < 2:
            per_pathology[path] = {
                'f1': float('nan'), 'f2': float('nan'),
                'sensitivity': float('nan'), 'specificity': float('nan'),
                'precision': float('nan')
            }
            continue
        
        try:
            tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
        except ValueError:
            tn = fp = fn = tp = 0
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        f1 = f1_score(yt, yp, zero_division=0)
        f2 = fbeta_score(yt, yp, beta=2.0, zero_division=0)
        
        per_pathology[path] = {
            'f1': float(f1),
            'f2': float(f2),
            'sensitivity': float(sensitivity),
            'specificity': float(specificity),
            'precision': float(precision),
            'tp': int(tp), 'fp': int(fp), 'tn': int(tn), 'fn': int(fn)
        }
        f1_list.append(f1); f2_list.append(f2)
        sens_list.append(sensitivity); spec_list.append(specificity); prec_list.append(precision)
    
    mean = {
        'f1': float(np.mean(f1_list)) if f1_list else float('nan'),
        'f2': float(np.mean(f2_list)) if f2_list else float('nan'),
        'sensitivity': float(np.mean(sens_list)) if sens_list else float('nan'),
        'specificity': float(np.mean(spec_list)) if spec_list else float('nan'),
        'precision': float(np.mean(prec_list)) if prec_list else float('nan'),
    }
    
    return {'per_pathology': per_pathology, 'mean': mean, 'threshold': threshold}


def log_message(msg):
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    line = f"[{timestamp}] {msg}"
    print(line)
    with open(os.path.join(LOG_DIR, 'mitigation_experiments.log'), 'a', encoding='utf-8') as f:
        f.write(line + '\n')

## 5. Label Smoothing

Define the binary cross-entropy loss variant that softens target labels to reduce overconfidence and limit sensitivity to incorrect labels.


In [20]:
class BCEWithLabelSmoothing(nn.Module):
    """BCE loss with label smoothing for multi-label classification."""
    def __init__(self, epsilon=0.1):
        super().__init__()
        self.epsilon = epsilon
        self.bce = nn.BCEWithLogitsLoss()
    
    def forward(self, logits, labels):
        # Smooth: 0 -> epsilon, 1 -> 1-epsilon
        smooth_labels = labels * (1 - self.epsilon) + 0.5 * self.epsilon
        return self.bce(logits, smooth_labels)


def train_epoch_label_smoothing(model, loader, criterion, optimizer, scaler, device):
    """Standard training with smoothed labels."""
    model.train()
    total_loss = 0.0
    for i, (images, labels) in enumerate(loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        if scaler is not None:
            with autocast('cuda'):
                logits = model(images)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
        total_loss += loss.item()
        if (i + 1) % 500 == 0:
            print(f"    Batch {i+1}/{len(loader)} loss={loss.item():.4f}")
    return total_loss / len(loader)

## 6. MixUp

Define the batch-level MixUp procedure that combines pairs of images and their labels using interpolation weights sampled from a beta distribution.


In [21]:
def mixup_data(images, labels, alpha=0.2):
    """Apply MixUp: blend pairs of samples in a batch."""
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    batch_size = images.size(0)
    idx = torch.randperm(batch_size, device=images.device)
    
    mixed_images = lam * images + (1 - lam) * images[idx]
    mixed_labels = lam * labels + (1 - lam) * labels[idx]
    return mixed_images, mixed_labels, lam


def train_epoch_mixup(model, loader, criterion, optimizer, scaler, device, alpha=0.2):
    """Training with MixUp augmentation."""
    model.train()
    total_loss = 0.0
    for i, (images, labels) in enumerate(loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        mixed_images, mixed_labels, lam = mixup_data(images, labels, alpha)
        
        optimizer.zero_grad(set_to_none=True)
        if scaler is not None:
            with autocast('cuda'):
                logits = model(mixed_images)
                loss = criterion(logits, mixed_labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(mixed_images)
            loss = criterion(logits, mixed_labels)
            loss.backward()
            optimizer.step()
        total_loss += loss.item()
        if (i + 1) % 500 == 0:
            print(f"    Batch {i+1}/{len(loader)} loss={loss.item():.4f}")
    return total_loss / len(loader)

## 7. Co-Teaching

Define the dual-network training procedure in which each model selects lower-loss samples for its peer according to the configured forget-rate schedule.


In [22]:
def train_epoch_coteaching(model1, model2, loader, optimizer1, optimizer2, scaler1, scaler2,
                            device, forget_rate=0.2, warmup=False):
    """Co-teaching: each network selects small-loss samples for the other."""
    model1.train()
    model2.train()
    total_loss1 = 0.0
    total_loss2 = 0.0
    bce_per_sample = nn.BCEWithLogitsLoss(reduction='none')
    
    for i, (images, labels) in enumerate(loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        # Forward pass through both networks
        if scaler1 is not None:
            with autocast('cuda'):
                logits1 = model1(images)
                logits2 = model2(images)
                # Per-sample loss (mean over pathologies)
                loss1_per = bce_per_sample(logits1, labels).mean(dim=1)
                loss2_per = bce_per_sample(logits2, labels).mean(dim=1)
        else:
            logits1 = model1(images)
            logits2 = model2(images)
            loss1_per = bce_per_sample(logits1, labels).mean(dim=1)
            loss2_per = bce_per_sample(logits2, labels).mean(dim=1)
        
        batch_size = images.size(0)
        
        if warmup:
            # First epoch: use all samples (standard training)
            selected1_mask = torch.ones(batch_size, dtype=torch.bool, device=device)
            selected2_mask = torch.ones(batch_size, dtype=torch.bool, device=device)
        else:
            # Select small-loss samples: keep (1 - forget_rate) fraction
            n_keep = int((1 - forget_rate) * batch_size)
            n_keep = max(1, n_keep)
            
            # Model 1's low-loss samples used to train Model 2
            _, idx1_keep = loss1_per.topk(n_keep, largest=False)
            selected2_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)
            selected2_mask[idx1_keep] = True
            
            # Model 2's low-loss samples used to train Model 1
            _, idx2_keep = loss2_per.topk(n_keep, largest=False)
            selected1_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)
            selected1_mask[idx2_keep] = True
        
        # Compute losses only on selected samples
        loss1 = loss1_per[selected1_mask].mean()
        loss2 = loss2_per[selected2_mask].mean()
        
        # Update model 1
        optimizer1.zero_grad(set_to_none=True)
        if scaler1 is not None:
            scaler1.scale(loss1).backward(retain_graph=True)
            scaler1.step(optimizer1)
            scaler1.update()
        else:
            loss1.backward(retain_graph=True)
            optimizer1.step()
        
        # Update model 2
        optimizer2.zero_grad(set_to_none=True)
        if scaler2 is not None:
            scaler2.scale(loss2).backward()
            scaler2.step(optimizer2)
            scaler2.update()
        else:
            loss2.backward()
            optimizer2.step()
        
        total_loss1 += loss1.item()
        total_loss2 += loss2.item()
        
        if (i + 1) % 500 == 0:
            print(f"    Batch {i+1}/{len(loader)} loss1={loss1.item():.4f} loss2={loss2.item():.4f}")
    
    return total_loss1 / len(loader), total_loss2 / len(loader)

## 8. Standard Training Utility

Define the standard single-model training routine reused where required, including within the confident-learning workflow.


In [23]:
def train_epoch_standard(model, loader, criterion, optimizer, scaler, device):
    """Standard training loop (used inside confident learning)."""
    model.train()
    total_loss = 0.0
    for i, (images, labels) in enumerate(loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        if scaler is not None:
            with autocast('cuda'):
                logits = model(images)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
        total_loss += loss.item()
        if (i + 1) % 500 == 0:
            print(f"    Batch {i+1}/{len(loader)} loss={loss.item():.4f}")
    return total_loss / len(loader)


@torch.no_grad()
def get_predictions_on_training(model, loader, device):
    """Get sigmoid probabilities on the training set for cleanlab."""
    model.eval()
    all_probs = []
    all_labels = []
    all_indices = []
    for images, labels, indices in loader:
        images = images.to(device, non_blocking=True)
        with autocast('cuda', enabled=device.type == 'cuda'):
            logits = model(images)
        probs = torch.sigmoid(logits).float().cpu().numpy()
        all_probs.append(probs)
        all_labels.append(labels.numpy())
        all_indices.append(indices.numpy())
    return np.concatenate(all_probs), np.concatenate(all_labels), np.concatenate(all_indices)


def identify_mislabeled_samples(probs, labels, prune_ratio=0.1):
    """Use cleanlab to identify likely mislabeled samples across all pathologies.
    
    Since we have multi-label, we apply cleanlab per-pathology and mark a sample
    as mislabeled if ANY of its pathology labels look wrong.
    """
    n_samples, n_classes = labels.shape
    mislabeled_mask = np.zeros(n_samples, dtype=bool)
    
    for c in range(n_classes):
        # For each pathology, run confident learning as binary classification
        # cleanlab expects 2D prob array [n_samples, n_classes=2]
        binary_probs = np.stack([1 - probs[:, c], probs[:, c]], axis=1)
        binary_labels = labels[:, c].astype(int)
        
        try:
            issues = find_label_issues(
                labels=binary_labels,
                pred_probs=binary_probs,
                return_indices_ranked_by='self_confidence',
                n_jobs=1,
            )
            # Take top prune_ratio fraction
            n_to_prune = int(prune_ratio * n_samples)
            if len(issues) > n_to_prune:
                issues = issues[:n_to_prune]
            mislabeled_mask[issues] = True
        except Exception as e:
            print(f"    cleanlab failed for pathology {c}: {e}")
    
    return mislabeled_mask

## 9. Run a Mitigation Experiment

Define the main experiment function that configures the selected method, trains the model or models, evaluates performance, and saves the resulting metrics.


In [24]:
def run_mitigation_experiment(method, noise_type, noise_rate, seed, train_config, mitigation_config):
    """Run one mitigation experiment. Returns results dict."""
    exp_name = f"{method}_{noise_type}_{int(noise_rate*100):02d}_seed{seed}"
    exp_start = time.time()
    log_message(f"START: {exp_name}")
    
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    
    train_csv = os.path.join(PROCESSED_DIR, f'train_{noise_type}_{int(noise_rate*100):02d}.csv')
    
    return_index = (method == 'confident_learning')
    train_dataset = CheXpertDataset(
        csv_path=train_csv, data_dir=RAW_DIR,
        pathologies=TARGET_PATHOLOGIES, transform=train_transform,
        return_index=return_index
    )
    train_loader = DataLoader(
        train_dataset, batch_size=train_config['batch_size'],
        shuffle=True, num_workers=0, pin_memory=True
    )
    
    best_path = os.path.join(MODELS_DIR, f'{exp_name}_best.pt')
    last_path = os.path.join(MODELS_DIR, f'{exp_name}_last.pt')  # NEW: per-epoch checkpoint
    history = {'train_loss': [], 'val_loss': [], 'val_auc': [], 'per_pathology_auc': []}
    best_val_auc = 0.0
    start_epoch = 1  # NEW: track where to resume from
    
    # ================================================================
    # METHOD-SPECIFIC TRAINING
    # ================================================================
    
    if method == 'label_smoothing':
        model = build_densenet121(len(TARGET_PATHOLOGIES), pretrained=True).to(device)
        criterion = BCEWithLabelSmoothing(epsilon=mitigation_config['label_smoothing_epsilon'])
        val_criterion = nn.BCEWithLogitsLoss()
        optimizer = optim.AdamW(model.parameters(), lr=train_config['learning_rate'],
                                weight_decay=train_config['weight_decay'])
        scaler = GradScaler('cuda') if device.type == 'cuda' else None
        
        if os.path.exists(last_path):
            try:
                ckpt = torch.load(last_path, weights_only=False)
                model.load_state_dict(ckpt['model_state_dict'])
                optimizer.load_state_dict(ckpt['optimizer_state_dict'])
                start_epoch = ckpt['epoch'] + 1
                history = ckpt.get('history', history)
                best_val_auc = ckpt.get('best_val_auc', 0.0)
                log_message(f"  Resumed from epoch {ckpt['epoch']}, best_val_auc={best_val_auc:.4f}")
            except Exception as e:
                log_message(f"  Failed to resume: {e}. Starting fresh.")
                start_epoch = 1
        
        epochs_no_improve = 0
        for epoch in range(start_epoch, train_config['num_epochs'] + 1):
            print(f"  Epoch {epoch}/{train_config['num_epochs']}")
            train_loss = train_epoch_label_smoothing(model, train_loader, criterion,
                                                       optimizer, scaler, device)
            val_loss, val_auc, per_path_auc = evaluate(model, val_loader, val_criterion,
                                                        device, TARGET_PATHOLOGIES)
            history['train_loss'].append(train_loss)
            history['val_loss'].append(val_loss)
            history['val_auc'].append(val_auc)
            history['per_pathology_auc'].append(per_path_auc)
            log_message(f"  Epoch {epoch}: train_loss={train_loss:.4f}, val_auc={val_auc:.4f}")
            
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'history': history,
                'best_val_auc': best_val_auc,
            }, last_path)
            
            if val_auc > best_val_auc:
                best_val_auc = val_auc
                epochs_no_improve = 0
                torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                            'val_auc': val_auc, 'per_pathology_auc': per_path_auc}, best_path)
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= train_config['early_stopping_patience']:
                    log_message(f"  Early stopping at epoch {epoch}")
                    break
    
    elif method == 'mixup':
        model = build_densenet121(len(TARGET_PATHOLOGIES), pretrained=True).to(device)
        criterion = nn.BCEWithLogitsLoss()
        optimizer = optim.AdamW(model.parameters(), lr=train_config['learning_rate'],
                                weight_decay=train_config['weight_decay'])
        scaler = GradScaler('cuda') if device.type == 'cuda' else None
        
        if os.path.exists(last_path):
            try:
                ckpt = torch.load(last_path, weights_only=False)
                model.load_state_dict(ckpt['model_state_dict'])
                optimizer.load_state_dict(ckpt['optimizer_state_dict'])
                start_epoch = ckpt['epoch'] + 1
                history = ckpt.get('history', history)
                best_val_auc = ckpt.get('best_val_auc', 0.0)
                log_message(f"  Resumed from epoch {ckpt['epoch']}, best_val_auc={best_val_auc:.4f}")
            except Exception as e:
                log_message(f"  Failed to resume: {e}. Starting fresh.")
                start_epoch = 1
        
        epochs_no_improve = 0
        for epoch in range(start_epoch, train_config['num_epochs'] + 1):
            print(f"  Epoch {epoch}/{train_config['num_epochs']}")
            train_loss = train_epoch_mixup(model, train_loader, criterion,
                                            optimizer, scaler, device,
                                            alpha=mitigation_config['mixup_alpha'])
            val_loss, val_auc, per_path_auc = evaluate(model, val_loader, criterion,
                                                        device, TARGET_PATHOLOGIES)
            history['train_loss'].append(train_loss)
            history['val_loss'].append(val_loss)
            history['val_auc'].append(val_auc)
            history['per_pathology_auc'].append(per_path_auc)
            log_message(f"  Epoch {epoch}: train_loss={train_loss:.4f}, val_auc={val_auc:.4f}")
            
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'history': history,
                'best_val_auc': best_val_auc,
            }, last_path)
            
            if val_auc > best_val_auc:
                best_val_auc = val_auc
                epochs_no_improve = 0
                torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                            'val_auc': val_auc, 'per_pathology_auc': per_path_auc}, best_path)
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= train_config['early_stopping_patience']:
                    log_message(f"  Early stopping at epoch {epoch}")
                    break
    
    elif method == 'coteaching':
        torch.manual_seed(seed)
        model1 = build_densenet121(len(TARGET_PATHOLOGIES), pretrained=True).to(device)
        torch.manual_seed(seed + 10000)
        model2 = build_densenet121(len(TARGET_PATHOLOGIES), pretrained=True).to(device)
        
        criterion = nn.BCEWithLogitsLoss()
        optimizer1 = optim.AdamW(model1.parameters(), lr=train_config['learning_rate'],
                                 weight_decay=train_config['weight_decay'])
        optimizer2 = optim.AdamW(model2.parameters(), lr=train_config['learning_rate'],
                                 weight_decay=train_config['weight_decay'])
        scaler1 = GradScaler('cuda') if device.type == 'cuda' else None
        scaler2 = GradScaler('cuda') if device.type == 'cuda' else None
        
        if os.path.exists(last_path):
            try:
                ckpt = torch.load(last_path, weights_only=False)
                model1.load_state_dict(ckpt['model1_state_dict'])
                model2.load_state_dict(ckpt['model2_state_dict'])
                optimizer1.load_state_dict(ckpt['optimizer1_state_dict'])
                optimizer2.load_state_dict(ckpt['optimizer2_state_dict'])
                start_epoch = ckpt['epoch'] + 1
                history = ckpt.get('history', history)
                best_val_auc = ckpt.get('best_val_auc', 0.0)
                log_message(f"  Resumed from epoch {ckpt['epoch']}, best_val_auc={best_val_auc:.4f}")
            except Exception as e:
                log_message(f"  Failed to resume: {e}. Starting fresh.")
                start_epoch = 1
        
        epochs_no_improve = 0
        warmup_epochs = mitigation_config['coteaching_warmup_epochs']
        forget_rate = mitigation_config['coteaching_forget_rate']
        
        for epoch in range(start_epoch, train_config['num_epochs'] + 1):
            is_warmup = epoch <= warmup_epochs
            print(f"  Epoch {epoch}/{train_config['num_epochs']} {'(warmup)' if is_warmup else ''}")
            loss1, loss2 = train_epoch_coteaching(
                model1, model2, train_loader, optimizer1, optimizer2,
                scaler1, scaler2, device, forget_rate=forget_rate, warmup=is_warmup
            )
            val_loss, val_auc, per_path_auc = evaluate(model1, val_loader, criterion,
                                                        device, TARGET_PATHOLOGIES)
            history['train_loss'].append((loss1 + loss2) / 2)
            history['val_loss'].append(val_loss)
            history['val_auc'].append(val_auc)
            history['per_pathology_auc'].append(per_path_auc)
            log_message(f"  Epoch {epoch}: loss1={loss1:.4f}, loss2={loss2:.4f}, val_auc={val_auc:.4f}")
            
            torch.save({
                'epoch': epoch,
                'model1_state_dict': model1.state_dict(),
                'model2_state_dict': model2.state_dict(),
                'optimizer1_state_dict': optimizer1.state_dict(),
                'optimizer2_state_dict': optimizer2.state_dict(),
                'history': history,
                'best_val_auc': best_val_auc,
            }, last_path)
            
            if val_auc > best_val_auc:
                best_val_auc = val_auc
                epochs_no_improve = 0
                torch.save({'epoch': epoch, 'model_state_dict': model1.state_dict(),
                            'val_auc': val_auc, 'per_pathology_auc': per_path_auc}, best_path)
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= train_config['early_stopping_patience']:
                    log_message(f"  Early stopping at epoch {epoch}")
                    break
        
        del model2, optimizer2, scaler2
        model = model1
    
    elif method == 'confident_learning':
        # Confident learning is a two-stage process. We track which stage
        # we are in via the checkpoint. Stage 1 = initial training on noisy data,
        # Stage 2 = retrain on cleaned data.
        
        stage = 1
        n_removed = 0
        cleaned_csv_path = os.path.join(PROCESSED_DIR, f'temp_cleaned_{exp_name}.csv')
        
        model = build_densenet121(len(TARGET_PATHOLOGIES), pretrained=True).to(device)
        criterion = nn.BCEWithLogitsLoss()
        optimizer = optim.AdamW(model.parameters(), lr=train_config['learning_rate'],
                                weight_decay=train_config['weight_decay'])
        scaler = GradScaler('cuda') if device.type == 'cuda' else None
        
        if os.path.exists(last_path):
            try:
                ckpt = torch.load(last_path, weights_only=False)
                model.load_state_dict(ckpt['model_state_dict'])
                optimizer.load_state_dict(ckpt['optimizer_state_dict'])
                start_epoch = ckpt['epoch'] + 1
                history = ckpt.get('history', history)
                best_val_auc = ckpt.get('best_val_auc', 0.0)
                stage = ckpt.get('stage', 1)
                n_removed = ckpt.get('n_removed', 0)
                log_message(f"  Resumed from stage {stage} epoch {ckpt['epoch']}")
            except Exception as e:
                log_message(f"  Failed to resume: {e}. Starting fresh.")
                start_epoch = 1
                stage = 1
        
        stage1_epochs = max(2, train_config['num_epochs'] - 2)
        
        # Train on noisy data (if not already done)
        if stage == 1:
            log_message("  Stage 1: Training initial model on noisy data")
            train_dataset_noindex = CheXpertDataset(
                csv_path=train_csv, data_dir=RAW_DIR,
                pathologies=TARGET_PATHOLOGIES, transform=train_transform,
                return_index=False
            )
            train_loader_noindex = DataLoader(train_dataset_noindex, batch_size=train_config['batch_size'],
                                              shuffle=True, num_workers=0, pin_memory=True)
            
            for epoch in range(start_epoch, stage1_epochs + 1):
                print(f"  Stage 1 Epoch {epoch}/{stage1_epochs}")
                train_loss = train_epoch_standard(model, train_loader_noindex, criterion,
                                                    optimizer, scaler, device)
                log_message(f"  Stage 1 Epoch {epoch}: train_loss={train_loss:.4f}")
                
                # Save per-epoch checkpoint
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'history': history,
                    'best_val_auc': best_val_auc,
                    'stage': 1,
                    'n_removed': 0,
                }, last_path)
            
            log_message("  Identifying mislabeled samples with cleanlab")
            eval_dataset = CheXpertDataset(
                csv_path=train_csv, data_dir=RAW_DIR,
                pathologies=TARGET_PATHOLOGIES, transform=eval_transform, return_index=True
            )
            eval_loader = DataLoader(eval_dataset, batch_size=train_config['batch_size'],
                                      shuffle=False, num_workers=0, pin_memory=True)
            probs, labels, indices = get_predictions_on_training(model, eval_loader, device)
            order = np.argsort(indices)
            probs = probs[order]
            labels = labels[order]
            
            mislabeled_mask = identify_mislabeled_samples(
                probs, labels, prune_ratio=mitigation_config['confident_learning_prune_ratio']
            )
            n_removed = int(mislabeled_mask.sum())
            log_message(f"  Cleanlab identified {n_removed}/{len(labels)} ({100*n_removed/len(labels):.1f}%) samples as mislabeled")
            
            cleaned_df = pd.read_csv(train_csv).iloc[~mislabeled_mask].reset_index(drop=True)
            cleaned_df.to_csv(cleaned_csv_path, index=False)
            
            # fresh model
            del model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            
            model = build_densenet121(len(TARGET_PATHOLOGIES), pretrained=True).to(device)
            optimizer = optim.AdamW(model.parameters(), lr=train_config['learning_rate'],
                                    weight_decay=train_config['weight_decay'])
            scaler = GradScaler('cuda') if device.type == 'cuda' else None
            
            stage = 2
            start_epoch = 1
            
            # Save transition checkpoint
            torch.save({
                'epoch': 0,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'history': history,
                'best_val_auc': 0.0,
                'stage': 2,
                'n_removed': n_removed,
            }, last_path)
        
        # Retrain on cleaned data
        log_message("  Stage 2: Retraining on cleaned data")
        cleaned_dataset = CheXpertDataset(
            csv_path=cleaned_csv_path, data_dir=RAW_DIR,
            pathologies=TARGET_PATHOLOGIES, transform=train_transform, return_index=False
        )
        cleaned_loader = DataLoader(cleaned_dataset, batch_size=train_config['batch_size'],
                                     shuffle=True, num_workers=0, pin_memory=True)
        
        epochs_no_improve = 0
        for epoch in range(start_epoch, train_config['num_epochs'] + 1):
            print(f"  Stage 2 Epoch {epoch}/{train_config['num_epochs']}")
            train_loss = train_epoch_standard(model, cleaned_loader, criterion,
                                                optimizer, scaler, device)
            val_loss, val_auc, per_path_auc = evaluate(model, val_loader, criterion,
                                                        device, TARGET_PATHOLOGIES)
            history['train_loss'].append(train_loss)
            history['val_loss'].append(val_loss)
            history['val_auc'].append(val_auc)
            history['per_pathology_auc'].append(per_path_auc)
            log_message(f"  Stage 2 Epoch {epoch}: train_loss={train_loss:.4f}, val_auc={val_auc:.4f}")
            
            # Save per-epoch checkpoint
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'history': history,
                'best_val_auc': best_val_auc,
                'stage': 2,
                'n_removed': n_removed,
            }, last_path)
            
            if val_auc > best_val_auc:
                best_val_auc = val_auc
                epochs_no_improve = 0
                torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                            'val_auc': val_auc, 'per_pathology_auc': per_path_auc,
                            'n_removed_by_cleanlab': int(n_removed)}, best_path)
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= train_config['early_stopping_patience']:
                    log_message(f"  Early stopping at epoch {epoch}")
                    break
        
        try:
            os.remove(cleaned_csv_path)
        except:
            pass
    
    else:
        raise ValueError(f"Unknown method: {method}")
    
    # ================================================================
    # FINAL EVALUATION ON OFFICIAL TEST SET
    # ================================================================
    log_message(f"  Evaluating best model on official test set")
    best_ckpt = torch.load(best_path, weights_only=False)
    model.load_state_dict(best_ckpt['model_state_dict'])
    test_criterion = nn.BCEWithLogitsLoss()
    test_loss, test_auc, test_per_path = evaluate(model, test_loader, test_criterion,
                                                    device, TARGET_PATHOLOGIES)
    
    exp_time = time.time() - exp_start
    results = {
        'experiment_name': exp_name,
        'method': method,
        'noise_type': noise_type,
        'noise_rate': noise_rate,
        'seed': seed,
        'best_epoch': best_ckpt['epoch'],
        'best_val_auc': float(best_val_auc),
        'best_val_per_pathology_auc': {k: float(v) for k, v in best_ckpt['per_pathology_auc'].items()},
        'test_mean_auc': float(test_auc),
        'test_per_pathology_auc': {k: float(v) for k, v in test_per_path.items()},
        'test_loss': float(test_loss),
        'history': history,
        'total_time_minutes': exp_time / 60,
        'config': train_config,
        'mitigation_config': mitigation_config,
    }
    
    results_path = os.path.join(RESULTS_DIR, f'{exp_name}.json')
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2)
    
    log_message(f"DONE: {exp_name} | test_auc={test_auc:.4f} | time={exp_time/60:.1f}min")
    
    # Remove transition checkpoint
    if os.path.exists(last_path):
        try:
            os.remove(last_path)
        except:
            pass
    
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    return results

## 10. Build and Resume the Experiment Queue

Construct all method × noise-condition × seed combinations and identify which experiments are already complete so unfinished runs can resume.


In [ ]:
# Use configuration from Cell 1 (do not override here)
method_order = MITIGATION_METHODS
noise_priority = NOISE_CONFIGS

# Build full experiment list
all_experiments = []
for method in method_order:
    for noise_type, noise_rate in noise_priority:
        for seed in SEEDS:
            exp_name = f"{method}_{noise_type}_{int(noise_rate*100):02d}_seed{seed}"
            all_experiments.append((method, noise_type, noise_rate, seed, exp_name))

# Check which are already done
completed = []
pending = []
for exp in all_experiments:
    method, noise_type, noise_rate, seed, exp_name = exp
    results_path = os.path.join(RESULTS_DIR, f'{exp_name}.json')
    if os.path.exists(results_path):
        completed.append(exp_name)
    else:
        pending.append(exp)

print(f"Total experiments: {len(all_experiments)}")
print(f"Already completed: {len(completed)}")
print(f"Pending:           {len(pending)}")

# Estimate remaining time
fast_methods_pending = sum(1 for e in pending if e[0] in ['label_smoothing', 'mixup'])
slow_methods_pending = sum(1 for e in pending if e[0] in ['coteaching', 'confident_learning'])
est_hours = (fast_methods_pending * 90 + slow_methods_pending * 180) / 60
print(f"\nEstimated remaining time: {est_hours:.1f} hours")

if completed:
    print(f"\nCompleted experiments:")
    for name in completed:
        print(f"  ✓ {name}")

if pending:
    print(f"\nFirst 5 pending experiments:")
    for exp in pending[:5]:
        print(f"  ○ {exp[4]}")

Total experiments: 72
Already completed: 36
Pending:           36

Estimated remaining time: 72.0 hours

Completed experiments:
  ✓ mixup_symmetric_20_seed132
  ✓ mixup_symmetric_20_seed456
  ✓ mixup_symmetric_20_seed789
  ✓ mixup_symmetric_40_seed132
  ✓ mixup_symmetric_40_seed456
  ✓ mixup_symmetric_40_seed789
  ✓ mixup_asymmetric_20_seed132
  ✓ mixup_asymmetric_20_seed456
  ✓ mixup_asymmetric_20_seed789
  ✓ mixup_asymmetric_40_seed132
  ✓ mixup_asymmetric_40_seed456
  ✓ mixup_asymmetric_40_seed789
  ✓ confident_learning_symmetric_20_seed132
  ✓ confident_learning_symmetric_20_seed456
  ✓ confident_learning_symmetric_20_seed789
  ✓ confident_learning_symmetric_40_seed132
  ✓ confident_learning_symmetric_40_seed456
  ✓ confident_learning_symmetric_40_seed789
  ✓ confident_learning_asymmetric_20_seed132
  ✓ confident_learning_asymmetric_20_seed456
  ✓ confident_learning_asymmetric_20_seed789
  ✓ confident_learning_asymmetric_40_seed132
  ✓ confident_learning_asymmetric_40_seed456
  ✓ c

: 

## 11. Execute Mitigation Experiments

Run each pending mitigation experiment and save results incrementally throughout the session.


In [ ]:
session_start = time.time()
log_message(f"=== MITIGATION SESSION START === Pending: {len(pending)}")

for i, (method, noise_type, noise_rate, seed, exp_name) in enumerate(pending):
    print("\n" + "=" * 80)
    print(f"EXPERIMENT {i+1}/{len(pending)}: {exp_name}")
    print("=" * 80)
    
    try:
        results = run_mitigation_experiment(
            method, noise_type, noise_rate, seed,
            TRAIN_CONFIG, MITIGATION_CONFIG
        )
        print(f"\n  Completed: test_auc={results['test_mean_auc']:.4f}")
        for path, auc in results['test_per_pathology_auc'].items():
            print(f"      {path:20s}: {auc:.4f}")
    except Exception as e:
        log_message(f"ERROR in {exp_name}: {e}")
        print(f"\n  FAILED: {e}")
        print(f"  Continuing with next experiment...")
        # Free GPU on failure
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        continue
    
    # Cooldown between experiments
    if i < len(pending) - 1:
        print(f"\n  Cooling down for {COOLDOWN_SECONDS}s...")
        for remaining in range(COOLDOWN_SECONDS, 0, -30):
            print(f"    {remaining}s remaining", end='\r')
            time.sleep(min(30, remaining))
        print("    Cooldown complete.       ")

session_time = time.time() - session_start
log_message(f"=== SESSION END === Total time: {session_time/3600:.2f} hours")
print(f"\n{'=' * 80}")
print(f"SESSION COMPLETE - Total time: {session_time/3600:.2f} hours")
print(f"{'=' * 80}")

[2026-08-16 09:02:16] === MITIGATION SESSION START === Pending: 36

EXPERIMENT 1/36: mixup_symmetric_05_seed132
[2026-08-16 09:02:16] START: mixup_symmetric_05_seed132
  Epoch 1/5
    Batch 500/5970 loss=0.5076
    Batch 1000/5970 loss=0.4995
    Batch 1500/5970 loss=0.5075
    Batch 2000/5970 loss=0.4808
    Batch 2500/5970 loss=0.5057
    Batch 3000/5970 loss=0.5807
    Batch 3500/5970 loss=0.4796
    Batch 4000/5970 loss=0.6036
    Batch 4500/5970 loss=0.5603
    Batch 5000/5970 loss=0.5684
    Batch 5500/5970 loss=0.6475
[2026-08-16 09:37:14]   Epoch 1: train_loss=0.5383, val_auc=0.8604
  Epoch 2/5
    Batch 500/5970 loss=0.4782
    Batch 1000/5970 loss=0.5674
    Batch 1500/5970 loss=0.5580
    Batch 2000/5970 loss=0.4944
    Batch 2500/5970 loss=0.5099
    Batch 3000/5970 loss=0.5805
    Batch 3500/5970 loss=0.4860
    Batch 4000/5970 loss=0.4986
    Batch 4500/5970 loss=0.5570
    Batch 5000/5970 loss=0.5260
    Batch 5500/5970 loss=0.5025
[2026-08-16 10:11:22]   Epoch 2: train_

## 12. Aggregate Mitigation Results

Load the completed result files into a combined table for later comparison of mitigation effectiveness.


In [4]:
# Load all completed mitigation results
rows = []
for fname in sorted(os.listdir(RESULTS_DIR)):
    if not fname.endswith('.json'):
        continue
    with open(os.path.join(RESULTS_DIR, fname)) as f:
        r = json.load(f)
    row = {
        'experiment': r['experiment_name'],
        'method': r['method'],
        'noise_type': r['noise_type'],
        'noise_rate': r['noise_rate'],
        'seed': r['seed'],
        'test_mean_auc': r['test_mean_auc'],
    }
    for path, auc in r['test_per_pathology_auc'].items():
        row[f'test_{path.replace(" ", "_")}'] = auc
    rows.append(row)

if rows:
    results_df = pd.DataFrame(rows)
    print(f"Loaded {len(results_df)} mitigation experiment results")
    results_df.to_csv(os.path.join(RESULTS_DIR, 'all_mitigation_results.csv'), index=False)
    
    print("\n=== Mean AUC by method, noise type, and rate (mean across seeds) ===")
    summary = results_df.groupby(['method', 'noise_type', 'noise_rate'])['test_mean_auc'].agg(['mean', 'std', 'count']).round(4)
    print(summary)
else:
    print("No results yet.")

Loaded 50 mitigation experiment results

=== Mean AUC by method, noise type, and rate (mean across seeds) ===
                                            mean     std  count
method             noise_type noise_rate                       
confident_learning asymmetric 0.20        0.8670  0.0152      3
                              0.40        0.8495  0.0079      3
                   symmetric  0.20        0.8723  0.0042      3
                              0.40        0.8156  0.0143      3
coteaching         asymmetric 0.20        0.8503  0.0119      3
                              0.40        0.8312  0.0130      3
                   symmetric  0.05        0.8526  0.0162      2
                              0.20        0.8573  0.0058      3
                              0.40        0.7943  0.0201      3
label_smoothing    asymmetric 0.20        0.8661  0.0108      3
                              0.40        0.8503  0.0129      3
                   symmetric  0.20        0.8756  0.0074  

## 13. Restore Normal Sleep Behaviour

Re-enable normal system sleep behaviour after all mitigation experiments have finished.


In [ ]:
allow_sleep()

Sleep prevention DISABLED
